# 实验三：Host侧体验——Host侧调用链分析与代码模板练习

## 小节概述

理解 Host 侧参数接收、约束检查、Tiling 数据组织、Kernel 启动和资源管理流程。

## 教程具体内容

以下内容围绕实验背景、任务准备、关键步骤和实验总结展开。


建议学时：2学时

## 实验任务

### 任务描述

---

原实验设计：

在华为云沙箱使用原生C++ API进行"破坏性刺探"，评测开发者体验。

修改后的设计：

本实验在CANNLAB平台完成，重点在于：

- 理解Host侧算子调用的完整流程

- 掌握tilingData的组织方式和计算方法

- 练习Host侧代码模板的编写

- 分析参数传递、约束检查、Kernel启动的调用链

修改原因：

Host侧代码开发是算子开发的核心环节，需要反复练习和调试。CANNLAB提供预置环境和样例代码，学生可以专注于代码逻辑理解，而不被环境配置干扰。

---

### 学习目标

小组分工：

- 开源软件开发工程师：主导Host侧代码编写和调用链梳理

- 开源社区布道师：记录代码理解障碍，绘制调用流程图

- 开源合规与安全工程师：审查代码规范和内存安全

学习资源路径：

- CANNLAB平台：使用预置的算子样例工程

- Host侧开发文档：CANN自定义算子开发手册

- 代码模板参考：CANNLAB中的sample工程

---

## 任务准备

### 前置知识

本实验需要提前学习以下相关知识：

- Host侧算子调度与Tiling计算基础
- Python编程基础
- Linux命令行操作基础

### 实验环境准备

本实验需要在以下环境中进行：

- **CANNLAB平台**：访问CANNLAB在线实验平台，获取预配置的算子开发环境
- **华为云沙箱**：进入华为云沙箱实验室，启动昇腾弹性云服务器

## 任务实施

### 实验要点

- 步骤一：理解Host侧算子调用的完整流程
- 步骤二：分析tilingData的组织方式
- 步骤三：梳理Host到Kernel的调用链
- 步骤四：练习Host侧代码模板编写
- 步骤五：区分业务逻辑与框架胶水代码

### 关键步骤

步骤一：理解Host侧算子调用的完整流程

在课程仓库内置的完整样例中查看 Host 侧代码结构：

`reference_practice/pytorch_online_inference_operator_optimize/src/add_custom_template/op_host`

重点阅读 `add_custom_template.cpp`，识别算子注册、Shape 推导和 Tiling 逻辑。后续代码单元会分别写出教学用的 Host 调用链和 Tiling 示例，不依赖预先存在的 `~/samples` 目录。

步骤二：分析tilingData的组织方式

编写tiling参数计算代码：

In [ ]:
%%writefile ./sinh_tiling_example.cpp
#include <cstdint>
#include <iostream>

// Host侧tiling数据结构定义
struct SinhTilingData {
    uint32_t total_length;   // 输入张量的总元素个数
    uint32_t block_dim;      // 并行执行的block数量
    uint32_t tile_num;       // 每个block内部的tile数量
    uint32_t tile_length;    // 每个tile处理的元素个数
};
// Tiling参数计算函数
void ComputeTilingData(uint32_t total_length, SinhTilingData& tiling) {
    tiling.total_length = total_length;
    
    // 根据输入规模和硬件能力计算并行参数
    // block_dim: 决定启动多少个并行核
    tiling.block_dim = 8;  // 示例值，实际应根据设备能力调整
    
    // tile_num: 每个核内部的数据分块数量
    tiling.tile_num = 8;   // 示例值，影响局部内存使用
    
    // tile_length: 每个tile处理的元素数量
    // 向上取整确保覆盖所有数据
    tiling.tile_length = (total_length + tiling.block_dim * tiling.tile_num - 1)
                       / (tiling.block_dim * tiling.tile_num);
    
    std::cout << "Tiling参数计算结果:" << std::endl;
    std::cout << "  total_length = " << tiling.total_length << std::endl;
    std::cout << "  block_dim = " << tiling.block_dim << std::endl;
    std::cout << "  tile_num = " << tiling.tile_num << std::endl;
    std::cout << "  tile_length = " << tiling.tile_length << std::endl;
}

步骤三：梳理Host到Kernel的调用链

绘制调用流程图：

Host侧入口 → 参数解析 → Shape检查 → Tiling计算 → TilingData写入Buffer → Kernel启动 → Kernel执行

步骤四：练习Host侧代码模板编写
在CANNLAB中编写完整的Host侧骨架代码：

In [ ]:
%%writefile ./host_template.cpp
#include <iostream>
#include "acl/acl.h"
#include "tiling/tiling_data.h"

// 步骤1: 定义tiling数据结构
struct MyOpTilingData {
    uint32_t total_length;
    uint32_t block_dim;
    uint32_t tile_length;
};

// 步骤2: 实现tiling计算函数
void ComputeTiling(uint32_t input_size, MyOpTilingData& tiling) {
    // 业务逻辑: 根据输入规模计算tiling参数
    tiling.total_length = input_size;
    tiling.block_dim = 8;
    tiling.tile_length = (input_size + tiling.block_dim - 1) / tiling.block_dim;
}

// 步骤3: 实现算子入口函数
int main() {
    // 3.1 初始化ACL环境
    aclError ret = aclInit(nullptr);
    if (ret != ACL_ERROR_NONE) {
        std::cerr << "aclInit failed: " << ret << std::endl;
        return -1;
    }
    
    // 3.2 创建context和stream
    aclrtContext context;
    aclrtStream stream;
    aclrtCreateContext(&context, 0);
    aclrtCreateStream(&stream);
    
    // 3.3 准备输入数据
    uint32_t input_size = 1024;
    void* input_dev = nullptr;
    void* output_dev = nullptr;
    aclrtMalloc(&input_dev, input_size * sizeof(float), ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc(&output_dev, input_size * sizeof(float), ACL_MEM_MALLOC_HUGE_FIRST);
    
    // 3.4 计算tiling参数
    MyOpTilingData tiling;
    ComputeTiling(input_size, tiling);
    
    // 3.5 启动kernel（概念性展示）
    // 实际工程中需要调用算子注册接口
    std::cout << "Host侧准备完成，tiling参数已计算" << std::endl;
    
    // 3.6 清理资源
    aclrtFree(input_dev);
    aclrtFree(output_dev);
    aclrtDestroyStream(stream);
    aclrtDestroyContext(context);
    aclFinalize();
    
    return 0;
}

步骤五：区分业务逻辑与框架胶水代码

## Host侧代码分类

### 业务逻辑（决定算子执行策略）

- 输入规模读取和shape检查

- block_dim和tile_length计算

- tiling参数组织和优化

- 数据类型和格式验证

### 框架胶水代码（服务于框架接入）

- ACL环境初始化和销毁

- context和stream管理

- 设备内存分配和释放

- 算子注册和调用接口

- 错误码处理和日志输出

### 对新手的影响

- 业务逻辑是算子开发的核心，需要深入理解

- 框架胶水代码可以通过模板和工具自动生成

- 过多的胶水代码可能掩盖业务逻辑，增加理解成本

## 任务拓展

《Host侧调用链分析报告.md》：报告内必须附带Host侧关键代码说明、一张"参数接收—约束检查—Tiling数据组织—Kernel启动"的调用流程图、至少3处框架胶水代码记录，以及业务逻辑与框架胶水代码的分类说明。报告还需区分Host侧代码中的业务逻辑与框架胶水代码，并总结其对新手理解算子调度过程的影响。

《Host侧代码模板》：包含可复用的Host侧骨架代码、tiling计算函数模板、资源管理代码模板。

## 实验总结

通过本次实验，完成了以下关键学习目标：

- 理解了Host侧算子调用的完整流程
- 掌握了tilingData的组织方式和计算方法
- 区分了业务逻辑与框架胶水代码

## 课后实践

请绘制 Host 侧调用链流程图，并区分业务逻辑与框架胶水代码。

## 参考答案

运行下面的代码单元查看参考调用链、代码分类和检查要点。


In [ ]:
!cat ./answer/01.03_answer.md